In [ ]:
%load_ext autoreload
%autoreload 2

import rastera
import geopandas as gpd
from rasterio.plot import show
from rastera.geo import Window

# Rastera

In [ ]:
bucket = "e84-earth-search-sentinel-data"
path = "sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B03.tif"
# https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a/items/S2B_T33TTG_20250703T100029_L2A?.language=en&.asset=asset-red
# uri1 = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B04.tif"
uri = "s3://e84-earth-search-sentinel-data/sentinel-2-c1-l2a/33/T/TG/2025/7/S2B_T33TTG_20250703T100029_L2A/B03.tif"
# https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a/items/S2B_T33TUG_20250703T100029_L2A?.language=en&.asset=asset-red
# uri2 = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/33/T/UG/2025/7/S2B_T33TUG_20250703T100029_L2A/B04.tif"
uri2 = "s3://e84-earth-search-sentinel-data/sentinel-2-c1-l2a/33/T/UG/2025/7/S2B_T33TUG_20250703T100029_L2A/B03.tif"

df_aoi = gpd.read_file("../data/sentinel2_rome_e84/rome_left_small.geojson")
df_aoi = df_aoi.to_crs(df_aoi.estimate_utm_crs())
print(df_aoi.crs.to_epsg())

bbox_utm = df_aoi.geometry.total_bounds
print(bbox_utm)

bands = [1]

df_aoi_shared = gpd.read_file("../data/sentinel2_rome_e84/rome_shared.geojson")
utm_crs = df_aoi_shared.estimate_utm_crs()
df_aoi_shared = df_aoi_shared.to_crs(utm_crs)
bbox_shared = df_aoi_shared.geometry.total_bounds

In [ ]:
src = await rastera.open(uri)
src

In [ ]:
# Alternative, share store
# from rastera import S3Store

# store = rastera.S3Store(bucket, region="us-west-2", skip_signature=True)
# src = await rastera.open(uri, store=store)
# src

In [ ]:
src.profile

In [ ]:
# Full img
img, profile = await src.read()
print(img.shape)
print(profile)
print(img.mean())

#show(img)

In [ ]:
# Read bbox
src = await rastera.open(uri)

data, profile = await src.read(bbox=bbox_utm, 
                              bbox_crs=utm_crs,
                              target_crs=32633,
                              target_resolution=20
                              )
print(data.shape)
print(profile)
print(data.mean())

In [ ]:
# Read window
window = Window(col_off=0, row_off=0, width=1000, height=2000)
data, profile = await src.read(window=window, target_resolution=20)

print(data.shape)
print(profile)
print(data.mean())

# Merged read

In [ ]:
# Before (sequential, separate stores):
#src1 = await rastera.open(uri)
#src2 = await rastera.open(uri2)

sources = await rastera.open([uri, uri2])
print(sources)

data, profile = await rastera.merge(sources, bbox=bbox_shared, bbox_crs=utm_crs, target_resolution=10)

print(data.shape)
print(profile)
print(data.mean())

In [ ]:
# rasterio mergeimport rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform
from rasterio.transform import from_bounds
import numpy as np
import rasterio

with rasterio.open(uri) as r1, rasterio.open(uri2) as r2:
    # Build the output transform from bbox + resolution
    res = 10
    dst_transform = from_bounds(*bbox_shared, 
                                width=int((bbox_shared[2] - bbox_shared[0]) / res),
                                height=int((bbox_shared[3] - bbox_shared[1]) / res))
    
    rio_img, rio_transform = merge(
        [r1, r2],
        bounds=tuple(bbox_shared),
        res=res,
    )
    
    print(rio_img.shape)
    print(rio_transform)
    print(rio_img.mean())